In [3]:
from pyaraucaria.obs_plan.obs_plan_parser import ObsPlanParser
from pyaraucaria.ob_validator import ObsValidator

print("ready")


ready


In [5]:
# txt = "OBJECT V496_Aql 19:08:20.77 -07:26:15.89 seq=1/V/20 \n WAIT ut=16:00:00"
# txt = "dupa "
# txt = "WAIT ut=16:00:00"
# txt = "WAIT sunrise=-10"
# txt = "WAIT sec=300"
# txt = "ZERO seq=15/Ic/0"
# txt = "DARK ZZ01 seq=10/V/300,10/Ic/200"
# txt = "DOMEFLAT AL3 seq=10/i/100"
# txt = "SKYFLAT HD23 alt=60.0 az=270.0 seq=10/r/20,10/V/30"
# txt = "SKYFLAT HD23 alt=60.0 az=270.0 seq=10/r/a"
# txt = "SKYFLAT HD24 seq=10/g/a,10/V/a"
# txt = "FOCUS NG31 12:12:12 20:20:20"
# txt = "OBJECT HD193901 20:23:35.8 -21:22:14.0 seq=1/V/300"
# txt = "OBJECT FF_Aql 18:58:14.75 17:21:39.29 seq=5/Ic/60,5/V/70"
# txt = "OBJECT V496_Aql 19:08:20.77 -07:26:15.89 seq=1/V/20"
# txt = "OBJECT V496_Aql seq=1/V/20"
# txt = "OBJECT HD23 alt=dupa az=270.0"
# txt = "OBJECT V496_Aql 19:08:20.77 "
# txt = "OBJECT V496_Aql 19:08:20.77 -07:26:15.89 alt=34 az=270.0 seq=1/V/20"
# txt = "WAIT"
# txt = "OBJECT HD193901 20:23:35.8 -21:22:14.0 seq=1/dupa/300"
# txt = "OBJECT HD193901 20:23:35.8 -21:22:14.0 seq=10x(1/Ic/300,/V/10)"
# txt = "OBJECT HD193901 20:23:35.8 -21:22:14.0 seq=seq=2/Ic/28,2/V/20,2/B/22,2/g/20,2/r/12,2/i/18,2/z/54"
# txt = "OBJECT HD193901 20:23:35.8 -21:22:14.0 seq=2/Ic/28,2/V/20,2/B/22,2/g/20,2/r/12,2/i/18,2/z/54"
# txt = "OBJECT HD193901 12:23:35.8 -63:45:00.54 seq=10x(1/Ic/300,3/V/10)"
# txt = "FOCUS HD193901 20:23:35.8 -21:22:14.0 seq=1/V/a"
txt = "OBJECT HD193901 seq=1/Ic/300"

# txt = "SKYFLAT HD23 alt=60.0 az=270.0 seq=10/r/a"


# Wczytanie schematów
BASE_SCHEMA = ObsValidator.load_schema("base_schema")
COMMAND_RULES = ObsValidator.load_schema("base_rules.yaml")

ob_tmp = ObsPlanParser.convert_from_string(txt)

if ob_tmp is None:
    print("\033[91m ❌ Parser error \033[0m")

else:
    ob = ObsValidator.convert_to_obdict(ob_tmp)


    # Walidacja
    validator = ObsValidator(BASE_SCHEMA, COMMAND_RULES)
    #ToDo list filtrow
    result = validator.validate_ob(ob,allowed_filters=["V","Ic","B","g","r","i","z"])

    print(result["valid"])
    print(result["result"])
    print(result["data"])
    #print(result["required"])
    #print(result["allowed"])

    prop = BASE_SCHEMA["properties"]

    for k in result["result"].keys():
        if not result["result"][k]:
            print(f' \033[91m ❌ {k}: \033[0m ')
            print(prop[k].get("description",""))
            print(prop[k].get("type",""))
            print(prop[k].get("enum",""))
            print(prop[k].get("examples",""))






True
{'command_name': True, 'seq': True, 'name': True}
{'command_name': 'OBJECT', 'seq': '1/Ic/300', 'name': 'HD193901'}


In [6]:
def merge_schemas(base: dict, extra: dict) -> dict:
    merged = base.copy()

    # properties
    merged.setdefault("properties", {})
    merged["properties"].update(extra.get("properties", {}))

    # required (jeśli masz)
    if "required" in base or "required" in extra:
        merged["required"] = list(set(base.get("required", []) + extra.get("required", [])))

    # additionalProperties – ostrożnie (tu przykład: AND)
    if "additionalProperties" in base or "additionalProperties" in extra:
        merged["additionalProperties"] = (
            base.get("additionalProperties", True)
            and extra.get("additionalProperties", True)
        )

    return merged

In [7]:
#txt = "BX_Del                  20:21:18.97        +18:26:16.29       seq=2/Ic/28,2/V/20,2/B/22,2/g/20,2/r/12,2/i/18,2/z/54               priority=5    cycle=0.08    ph_mk=1/Ic/0.006    ph_start=0.341    ph_end=0.567    uobi=4d5a6db1    sciprog=t2cep_mw    pi=pwielgorski    tag=t2cep,ascending_branch    P=1.091787    mV=12.31    "

txt = 'VV_Cru                  12:23:31.78        -64:28:15.2        seq=2/B/30,2/V/8,2/Ic/3.5,2/g/16,2/r/3.5,2/i/3.5,2/z/15             priority=dupa    cycle=0.167    ph_mk=2/V/0.02    sciprog=ccep_mw    pi=ghajdu    tag=ccep,random_phase    P=6.1210726    comment="G=11.385, Fshortmissing" '

#txt = 'BD-04_3206              12:06:20.8         -04:54:50          seq=2/B/4,2/V/2.5,2/Ic/3,2/y_s/7.5,2/v_s/20,2/b_s/20                priority=2    cycle=0.25    ph_mk=2/V/0.015    uobi=45eed6f1    sciprog=deb_sbcr    pi=dgraczyk    tag=random_phase    P=5.422527    hjd0=2457618.388    mV=10.8    comment="out of ecl, gaia, vby/BVI?" '

txt = f'OBJECT {txt}'

# Wczytanie schematów
BASE_SCHEMA = ObsValidator.load_schema("base_schema")
COMMAND_RULES = ObsValidator.load_schema("base_rules.yaml")

TPG_SCHEMA = ObsValidator.load_schema("tpg_schema.yaml")

ob_tmp = ObsPlanParser.convert_from_string(txt)

if ob_tmp is None:
    print("\033[91m ❌ Parser error \033[0m")

else:
    ob = ObsValidator.convert_to_obdict(ob_tmp)

    SCHEMA = merge_schemas(BASE_SCHEMA, TPG_SCHEMA)

    # Walidacja
    validator = ObsValidator(SCHEMA, COMMAND_RULES)
    result = validator.validate_ob(ob,allowed_filters=["V","Ic","B","g","r","i","z"])

    print(result["valid"])
    print(result["result"])
    print(result["data"])
    #print(result["required"])
    #print(result["allowed"])

    prop = TPG_SCHEMA["properties"]

    for k in result["result"].keys():
        if not result["result"][k]:
            print(f' \033[91m ❌ {k}: \033[0m ')
            print(prop[k].get("description",""))
            print(prop[k].get("type",""))
            print(prop[k].get("enum",""))
            print(prop[k].get("examples",""))


False
{'command_name': True, 'seq': True, 'priority': False, 'cycle': True, 'ph_mk': True, 'sciprog': True, 'pi': True, 'tag': True, 'P': True, 'comment': True, 'name': True, 'ra': True, 'dec': True}
{'command_name': 'OBJECT', 'seq': '2/B/30,2/V/8,2/Ic/3.5,2/g/16,2/r/3.5,2/i/3.5,2/z/15', 'priority': 'dupa', 'cycle': 0.167, 'ph_mk': '2/V/0.02', 'sciprog': 'ccep_mw', 'pi': 'ghajdu', 'tag': 'ccep,random_phase', 'P': 6.1210726, 'comment': '"G=11.385, Fshortmissing"', 'name': 'VV_Cru', 'ra': '12:23:31.78', 'dec': '-64:28:15.2'}
  ❌ priority:  
If <0, object will never be allocated; higher numbers = higher priority
number


